# Praktikum 4 - Perbandingan ANNOY, FAISS, dan HNSW pada 1 Juta Data 5D

**Nama:** Mohammad Al Hafis Hidayatulloh  
**NIM:** 254107023005  
**Mata Kuliah:** RTI235004 - Pembelajaran Mesin  
**Dosen Pengampu:** Usman Nurhasan, S.Kom., M.T.

---

## Percobaan Utama: Perbandingan Build Time, Query Time, dan Tetangga Terdekat (`Euclidean / L2`)

Pada praktikum keempat ini saya membandingkan secara langsung ketiga pustaka ANN (**Annoy**, **FAISS**, dan **HNSW**) menggunakan dataset yang sama persis yaitu `1.000.000` vektor berdimensi `5D` untuk mencari `k = 10` tetangga terdekat.

In [1]:
import numpy as np
import time
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
np.random.seed(42)
n_data = 1_000_000
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time_ann = time.time() - start

start = time.time()
neighbors_ann = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time_ann = time.time() - start

print("Build time:", round(build_time_ann, 4), "detik")
print("Query time:", round(query_time_ann, 6), "detik")
print("Neighbors:", neighbors_ann[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time_faiss = time.time() - start

start = time.time()
distances_faiss, indices_faiss = faiss_index.search(query, k)
query_time_faiss = time.time() - start

print("Build time:", round(build_time_faiss, 4), "detik")
print("Query time:", round(query_time_faiss, 6), "detik")
print("Neighbors:", indices_faiss[0][:5].tolist(), "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time_hnsw = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels_hnsw, distances_hnsw = hnsw_index.knn_query(query, k=k)
query_time_hnsw = time.time() - start

print("Build time:", round(build_time_hnsw, 4), "detik")
print("Query time:", round(query_time_hnsw, 6), "detik")
print("Neighbors:", labels_hnsw[0][:5].tolist(), "...")

=== Annoy ===
Build time: 2.8388 detik
Query time: 0.003836 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.0033 detik
Query time: 0.002252 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...

=== HNSW (hnswlib) ===
Build time: 34.2632 detik
Query time: 0.000149 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...


## Eksperimen pada Metrik Jarak Berbeda (`Angular / Cosine / Inner Product`)

Selanjutnya saya menjalankan perbandingan yang sama menggunakan metrik kemiripan berbasis sudut arah vektor (`angular` pada Annoy, `IndexFlatIP` dengan normalisasi L2 pada FAISS, dan `space='cosine'` pada HNSW) serta merangkum seluruh hasilnya ke dalam tabel perbandingan.

In [2]:
# Pengujian metrik Angular / Cosine pada dataset 1 juta x 5D yang sama
# 1. Annoy Angular
ann_ang = AnnoyIndex(dim, 'angular')
t0 = time.time()
for i in range(n_data):
    ann_ang.add_item(i, X[i])
ann_ang.build(10)
bt_ann_ang = time.time() - t0

t0 = time.time()
nb_ann_ang = ann_ang.get_nns_by_vector(query[0], k, include_distances=True)
qt_ann_ang = time.time() - t0

# 2. FAISS Inner Product (Cosine via L2 normalization)
X_norm = X.copy()
q_norm = query.copy()
faiss.normalize_L2(X_norm)
faiss.normalize_L2(q_norm)
faiss_ip = faiss.IndexFlatIP(dim)
t0 = time.time()
faiss_ip.add(X_norm)
bt_faiss_ang = time.time() - t0

t0 = time.time()
_, ind_faiss_ang = faiss_ip.search(q_norm, k)
qt_faiss_ang = time.time() - t0

# 3. HNSW Cosine
hnsw_cos = hnswlib.Index(space='cosine', dim=dim)
t0 = time.time()
hnsw_cos.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_cos.add_items(X)
bt_hnsw_ang = time.time() - t0
hnsw_cos.set_ef(50)

t0 = time.time()
lbl_hnsw_ang, _ = hnsw_cos.knn_query(query, k=k)
qt_hnsw_ang = time.time() - t0

df_tabel_p4 = pd.DataFrame([
    {
        'Model / Library': 'Annoy (10 trees)',
        'Distance Metric': 'Euclidean',
        'Build Time (detik)': round(build_time_ann, 4),
        'Query Time (ms)': round(query_time_ann * 1000, 4),
        'Top-5 Neighbors': str(neighbors_ann[0][:5])
    },
    {
        'Model / Library': 'FAISS (IndexFlatL2)',
        'Distance Metric': 'L2 (Euclidean)',
        'Build Time (detik)': round(build_time_faiss, 4),
        'Query Time (ms)': round(query_time_faiss * 1000, 4),
        'Top-5 Neighbors': str(indices_faiss[0][:5].tolist())
    },
    {
        'Model / Library': 'HNSW (hnswlib)',
        'Distance Metric': 'L2 (Euclidean)',
        'Build Time (detik)': round(build_time_hnsw, 4),
        'Query Time (ms)': round(query_time_hnsw * 1000, 4),
        'Top-5 Neighbors': str(labels_hnsw[0][:5].tolist())
    },
    {
        'Model / Library': 'Annoy (10 trees)',
        'Distance Metric': 'Angular',
        'Build Time (detik)': round(bt_ann_ang, 4),
        'Query Time (ms)': round(qt_ann_ang * 1000, 4),
        'Top-5 Neighbors': str(nb_ann_ang[0][:5])
    },
    {
        'Model / Library': 'FAISS (IndexFlatIP + Norm)',
        'Distance Metric': 'Cosine / Inner Product',
        'Build Time (detik)': round(bt_faiss_ang, 4),
        'Query Time (ms)': round(qt_faiss_ang * 1000, 4),
        'Top-5 Neighbors': str(ind_faiss_ang[0][:5].tolist())
    },
    {
        'Model / Library': 'HNSW (hnswlib)',
        'Distance Metric': 'Cosine',
        'Build Time (detik)': round(bt_hnsw_ang, 4),
        'Query Time (ms)': round(qt_hnsw_ang * 1000, 4),
        'Top-5 Neighbors': str(lbl_hnsw_ang[0][:5].tolist())
    }
])
df_tabel_p4

,Model / Library,Distance Metric,Build Time (detik),Query Time (ms),Top-5 Neighbors
0,Annoy (10 trees),Euclidean,2.8388,3.8357,"[901095, 495168, 561017, 163986, 185734]"
1,FAISS (IndexFlatL2),L2 (Euclidean),0.0033,2.2516,"[901095, 495168, 561017, 163986, 185734]"
2,HNSW (hnswlib),L2 (Euclidean),34.2632,0.1490,"[901095, 495168, 561017, 163986, 185734]"
3,Annoy (10 trees),Angular,2.6989,7.1809,"[653774, 135588, 561017, 901095, 211473]"
4,FAISS (IndexFlatIP + Norm),Cosine / Inner Product,0.0053,1.2438,"[653774, 135588, 561017, 901095, 211473]"
5,HNSW (hnswlib),Cosine,28.1898,0.1493,"[653774, 135588, 561017, 901095, 211473]"


### Analisis Hasil Praktikum 4

Dari tabel perbandingan ketiga pustaka pada 1 juta data berdimensi 5 di atas, terlihat jelas kompromi (*trade-off*) antara waktu pembangunan indeks (*build time*) dan kecepatan pencarian (*query time*):

1. **Ketepatan Hasil Pencarian:** Baik pada metrik `Euclidean / L2` maupun `Angular / Cosine`, ketiga metode berhasil menemukan 5 indeks tetangga terdekat yang sama persis, yaitu `[901095, 495168, 561017, 163986, 185734]` untuk jarak Euclidean dan `[653774, 135588, 561017, 901095, 211473]` untuk jarak Angular/Cosine.
2. **Perbandingan Build Time:** `FAISS IndexFlat` memiliki waktu *build* paling instan (kurang dari `0.01` detik) karena `IndexFlat` hanya menyalin array vektor ke memori tanpa membangun struktur pohon maupun graf. `Annoy` membutuhkan waktu menengah (sekitar `2.5` detik) untuk mempartisi 1 juta titik ke dalam 10 *random projection trees*. Sebaliknya, `HNSW` membutuhkan waktu *build* paling lama (sekitar `31 - 35` detik) karena harus menyusun koneksi graf *small-world* bertingkat untuk 1 juta simpul dengan parameter `ef_construction = 200` dan `M = 16`.
3. **Perbandingan Query Time:** Pengorbanan waktu *build* pada `HNSW` terbayar penuh saat proses pencarian (*query*), di mana `HNSW` menjadi yang paling cepat (hanya sekitar `0.18 - 0.48 ms`), jauh mengungguli `FAISS IndexFlat` dan `Annoy`.